# Process Forcings (`case.process_forcings`)

The final step: generate the forcing files your `configure_forcings` call (previous
notebook) declared. This notebook covers the normal one-shot path, and the
offline/iterative path for long or large runs where generating everything at once is
too slow or resource-intensive.

This notebook covers:
- [Section 1](#section-1-normal-one-shot-processing) — the normal, one-shot path
- [Section 2](#section-2-offline-or-iterative-processing-for-long-runs) — offline/iterative processing (`config.json`, `crocodash process`) for long runs

📖 [CrocoDash process_forcings docs](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/3b_process_forcings.html) · [CLI reference](advanced/cli_workflow.md) · [regional-mom6 docs](https://regional-mom6.readthedocs.io/en/latest/) (OBC regridding internals)

## Section 1: Normal, One-Shot Processing

In this step, we call the `process_forcings` method of CrocoDash to cut out and
interpolate the initial condition as well as all boundaries. CrocoDash also updates
MOM6 runtime parameters and CESM xml variables accordingly.

In [ ]:
case.process_forcings()

In [ ]:
print("You can now build and run your case at", caseroot)

## Section 2: Offline or Iterative Processing (for Long Runs)

Often, generating OBC datasets can be done all in one shot, but in longer and larger
cases (like running the Northwest Atlantic for a year) we need to start iterating
through the generation. Generating open boundary condition (OBC) data is essential for
the entire model runtime but can be time-consuming and resource-intensive.

The Extract Forcings Workflow in CrocoDash helps manage this by breaking data access
into smaller, more manageable pieces — though the piecewise/chunked download+regrid
pattern described below only applies to boundary conditions (`--bc`). The other forcing
types (initial conditions, tides, chlorophyll, runoff, BGC) each run as a single call
regardless of run length.

### Extract Forcings Workflow Overview

The workflow is enabled in all cases. When `configure_forcings` is called, it creates
an `extract_forcings/` folder under your input directory and writes a configuration
file (`config.json`) there describing the boundary condition and other forcing files
you need. You trigger the workflow from the shell with `crocodash process` (see the
[CLI reference](advanced/cli_workflow.md)).

#### Folder Structure

`inputdir/extract_forcings/` only holds data and config *for this case* — the actual
processing code (`driver.py` and the modules it calls) lives in the installed
CrocoDash package and is never copied per-case:

- **config.json** – written by `configure_forcings()`; the region-specific
  requirements and run parameters `crocodash process`/`run_workflow()` read back.
- **raw_data/** – downloaded-but-unprocessed chunks, created during `process_forcings()`
  for both the initial condition (`--ic`) and boundary conditions (`--bc`).
- **regridded_data/** – regridded OBC chunks, created during `process_forcings()` for
  boundary conditions (`--bc`) only.

Both cache directories let a re-run skip work that already succeeded instead of
starting over from scratch.

#### What Each Component Actually Does

`driver.py`'s `run_workflow()` (what `crocodash process` calls under the hood — its own
module docstring says as much directly) dispatches to one function per component you
request:

| CLI flag | Function | Chunked? |
|---|---|---|
| `--bc` | `obc.process_obc_conditions` | Yes — in `step`-day windows, cached in `raw_data/`/`regridded_data/` |
| `--ic` | `initial_condition.process_initial_condition` | No (but still caches raw downloads in `raw_data/`) |
| `--tides` | `tides.process_tides` | No |
| `--chl` | `chlorophyll.process_chl` | No |
| `--runoff` | `runoff.generate_rof_ocn_map` | No |
| `--bgcic` / `--bgcironforcing` / `--bgcrivernutrients` | `bgc.process_bgc_ic` / `process_bgc_iron_forcing` / `process_river_nutrients` | No |

#### How to Use

1. Identify and allocate available computing resources — on Derecho, get a starter
   PBS submission script with `crocodash template --output submit_forcings.pbs --kind pbs --machine derecho`
   (see the [CLI reference](advanced/cli_workflow.md#submitting-forcing-extraction-to-a-pbs-queue)).
2. For long boundary-condition runs specifically, adjust the `step` parameter (inside
   `config.json`'s `conditions.outputs` block) — it defaults to the **entire** date
   range (no chunking) unless you shrink it. Smaller steps mean more, cheaper chunks;
   this is the main knob for fitting a long OBC run into a walltime or memory limit.
3. Run `crocodash process` — you can rerun with a different `--skip` list if only some
   forcings changed, without redoing everything (see below).

See the CrocoDash docs' [Process Forcings](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/3b_process_forcings.html)
page for more detail.

### Running `crocodash process`

`inputdir/extract_forcings/` holds only `config.json` and (once you've run this at
least once) the `raw_data/`/`regridded_data/` caches — not a copy of the processing
code itself, so running it always needs CrocoDash installed: `crocodash process` is a
CLI entry point CrocoDash ships, not a standalone script sitting in that folder.

Run it from inside the `extract_forcings` directory — `config.json` is auto-detected
from the current directory:

```bash
cd <inputdir>/glorys/extract_forcings
crocodash process --all
```

Or from anywhere, with `--caseroot` or a direct `--config` path:

```bash
crocodash process --caseroot <caseroot> --all
crocodash process --config <inputdir>/glorys/extract_forcings/config.json --all
```

Especially consider adjusting the data-download function in `config.json`: on Derecho,
use the RDA reader (`get_glorys_data_from_rda`); on a local machine, use a GLORYS
API/CLI function instead (see [Configure Forcings, Section 2](configure_forcings.ipynb#section-2-switching-data-products)
for the full list of access functions). Change it by editing the `function_name` field
under `config.json`'s `conditions.inputs` block.

Use `--help` (or `crocodash process --help`) to see all available flags, including
running only specific forcings (`--ic --tides --runoff`) or skipping ones you don't
need (`--all --skip bgcic`). There's no bare `--bgc` flag — BGC is three separate
components (`--bgcic`, `--bgcironforcing`, `--bgcrivernutrients`) you enable
individually.

## Next steps

- [Interior OBC Segments](advanced/interior_obc_segments.ipynb) and
  [Nesting](advanced/nesting_demo.ipynb) — advanced domain/boundary topics
- [Use Cases](use_cases/three_boundary.ipynb) — real-world, end-to-end configurations